# Chess Piece Detection: YOLOv8 Training

AIML339, Anjola Sigbeku (300665808), Victoria University of Wellington.

Trains YOLOv8 on 4,888 rendered chess positions from the chesscog dataset
and evaluates performance separately for each camera angle and lighting
condition.

**Before running:** set the runtime to GPU via
*Runtime -> Change runtime type -> T4 GPU*.

The dataset is downloaded fresh each session rather than kept on Drive.
It is about 7 GB extracted, which would consume most of a free 15 GB Drive
quota, and Colab's connection fetches it in a few minutes anyway. Only the
trained weights and results are written to Drive.


## 1. Confirm the GPU


In [ ]:
!nvidia-smi

import torch
print()
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit(
        'No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.'
    )


## 2. Mount Drive

Only weights and result tables go here, so it stays small.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/AIML339/chess-detection')
RUNS = DRIVE / 'runs'
REPORTS = DRIVE / 'reports'
for d in (RUNS, REPORTS):
    d.mkdir(parents=True, exist_ok=True)
print('runs    ->', RUNS)
print('reports ->', REPORTS)


## 3. Clone the repository


In [ ]:
REPO = 'https://github.com/Logic911e/chess-piece-detection.git'

%cd /content
![ -d chess-piece-detection ] && rm -rf chess-piece-detection
!git clone -q $REPO
%cd /content/chess-piece-detection
!git log --oneline -1


## 4. Install dependencies


In [ ]:
!pip install -q ultralytics osfclient

import ultralytics
print('ultralytics', ultralytics.__version__)


## 5. Download the dataset

About 6.9 GB of archives from OSF, extracting to 4,888 images. Downloads
resume if interrupted, so re-running this cell is safe.


In [ ]:
!python src/download_synth.py --clean


## 6. Convert to YOLO format and audit

The audit exits non-zero on any hard error, so a bad conversion stops here
rather than surfacing as a confusing training failure later.


In [ ]:
!python src/synth_to_yolo.py --src data/raw/render --dest data/processed/synth
!python src/inspect_data.py --data data/processed/synth --json reports/audit.json


## 7. Train

yolov8s for 50 epochs. Checkpoints land on Drive every epoch, so a
disconnect costs one epoch rather than the run. To resume after a drop,
re-run every cell above, then run the resume cell below instead of this one.


In [ ]:
!python src/train_yolo.py \
    --data data/processed/synth/data.yaml \
    --project '/content/drive/MyDrive/AIML339/chess-detection/runs' \
    --name yolov8s-synth \
    --model yolov8s.pt \
    --epochs 50 \
    --batch 16


### Resume an interrupted run

Only needed if training was cut short.


In [ ]:
# !python src/train_yolo.py --resume \
#     --project '/content/drive/MyDrive/AIML339/chess-detection/runs'


## 8. Stratified evaluation

Reports mAP for each camera angle band and lighting mode separately.
This is the result that says *which conditions* the model struggles with,
rather than just reporting one aggregate number.


In [ ]:
WEIGHTS = RUNS / 'yolov8s-synth' / 'weights' / 'best.pt'
assert WEIGHTS.is_file(), f'No weights at {WEIGHTS}'
print(WEIGHTS)


In [ ]:
!python src/evaluate.py \
    --weights '$WEIGHTS' \
    --data data/processed/synth \
    --split test


## 9. Copy results to Drive


In [ ]:
import shutil

for name in ('eval_test.json', 'eval_test.csv', 'audit.json'):
    src = Path('reports') / name
    if src.is_file():
        shutil.copy2(src, REPORTS / name)
        print('saved', REPORTS / name)
    else:
        print('missing', src)


## 10. Results at a glance


In [ ]:
import json, pandas as pd

with open('reports/eval_test.json') as fh:
    payload = json.load(fh)

rows = []
for stratum, m in payload['metrics'].items():
    rows.append({
        'stratum': stratum,
        'images': payload['counts'].get(stratum),
        **{k: round(v, 4) for k, v in m.items()},
    })
display(pd.DataFrame(rows))

overall = payload['per_class_map'].get('overall', {})
if overall:
    pc = pd.DataFrame(
        sorted(overall.items(), key=lambda kv: kv[1]),
        columns=['class', 'mAP50-95'],
    )
    display(pc)
